# Uncertainty-TTA — Baseline Training on Kaggle / Colab

This notebook trains one MedMNIST dataset using the **shared pipeline** from the repo. It produces the same outputs as running `python -m scripts.train_baseline` locally — same hyperparameters, same metric definitions, same checkpoint format — but with the GPU you get for free on Kaggle.

**Before you start:**
1. **Kaggle**: Settings → Accelerator → **GPU T4 x2** (or P100). **Colab**: Runtime → Change runtime type → **T4 GPU**.
2. Set `DATASET` in the next cell to your assigned dataset key.
3. Run all cells top-to-bottom.
4. After it finishes, paste the printed tracker line into Sheet 1️⃣ Baselines, then open a PR with your `results/{dataset}_baseline.json`.

In [ ]:
# ───────────────────────────────────────────────────────────────
# CONFIG — set this to your dataset key, then run all cells
# ───────────────────────────────────────────────────────────────
# Options: pathmnist, dermamnist, pneumoniamnist, breastmnist, bloodmnist, organamnist

DATASET    = "dermamnist"   # ← change me
EPOCHS     = 30
BATCH_SIZE = 64
LR         = 1e-4
SEED       = 42

REPO_URL    = "https://github.com/mustafaerensoyhan/uncertainty-tta-medmnist.git"
REPO_BRANCH = "main"

## 1. Verify GPU

In [ ]:
import torch
print('PyTorch    :', torch.__version__)
print('CUDA avail :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU        :', torch.cuda.get_device_name(0))
else:
    print('⚠ No GPU detected. Enable it in Settings → Accelerator before continuing.')

## 2. Clone the repo and install dependencies

Kaggle and Colab come with PyTorch + scikit-learn + matplotlib + pandas already installed — we only need the `medmnist` package on top.

In [ ]:
import os
if not os.path.isdir('uncertainty-tta-medmnist'):
    !git clone --branch {REPO_BRANCH} {REPO_URL}
%cd uncertainty-tta-medmnist
!pip install -q medmnist==3.0.2

## 3. (Optional) Visual sanity check — preview 6 training samples

Confirms the data loader is working and shows you what the (normalized then denormalized) images look like for your assigned modality. Skip if you're in a hurry.

In [ ]:
import sys
sys.path.insert(0, '.')  # so we can import src.* directly
from src.data import get_dataset
from src.config import get_config
import matplotlib.pyplot as plt

cfg = get_config(DATASET)
train_ds = get_dataset(cfg, split='train', img_size=64)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def denorm(t):
    t = t.clone()
    for c in range(3):
        t[c] = t[c] * IMAGENET_STD[c] + IMAGENET_MEAN[c]
    return t.clamp(0, 1)

fig, axes = plt.subplots(1, 6, figsize=(12, 2.2))
for i, ax in enumerate(axes):
    img, lbl = train_ds[i]
    ax.imshow(denorm(img).permute(1, 2, 0).numpy())
    ax.set_title(f'y={int(lbl.squeeze())}', fontsize=9)
    ax.axis('off')
plt.suptitle(f'{cfg.modality} — first 6 train samples', y=1.05, fontsize=10)
plt.tight_layout(); plt.show()
print(f'train / val / test sizes will be loaded by the script below.')

## 4. Train the baseline

Calls `scripts/train_baseline.py` — the exact same entry point everyone else uses locally. First run downloads the dataset (~20–190 MB depending on which one) and the ResNet-18 ImageNet weights (~45 MB). Subsequent runs use the local cache.

The script will:
1. Download + load the dataset
2. Train ResNet-18 for 30 epochs (Adam, lr=1e-4)
3. Save the best-val checkpoint and a per-epoch training log
4. Evaluate on the test split (Accuracy, AUC, ECE, NLL)
5. Save a reliability diagram and training curves
6. Print a copy-pasteable tracker row
7. Check the result is within ±2% of the published MedMNIST benchmark

In [ ]:
!python -m scripts.train_baseline \
    --dataset {DATASET} \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --lr {LR} \
    --seed {SEED} \
    --num-workers 2

## 5. Inspect the saved metrics JSON

In [ ]:
import json
with open(f'results/{DATASET}_baseline.json') as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2))

## 6. Show the training curves inline

In [ ]:
from IPython.display import Image, display
display(Image(filename=f'figures/curves/{DATASET}_train_curves.png'))

## 7. Show the reliability diagram inline

Points below the diagonal mean the model is **overconfident** (it says e.g. 90% but is actually right 70% of the time). Points above mean **underconfident**. A baseline should sit near the diagonal at high confidence; the TTA experiments in Phase 2/3 will measure how the curve changes.

In [ ]:
display(Image(filename=f'figures/reliability/{DATASET}_baseline.png'))

## 8. Optional: peek at the per-epoch training log

In [ ]:
import pandas as pd
log = pd.read_csv(f'results/{DATASET}_train_log.csv')
print(log.to_string(index=False))

---

# Phase 2 — Standard TTA

The cells below run **standard equal-weight TTA** on the same dataset, reusing the checkpoint that Phase 1 just trained (it's still in `checkpoints/` from the training run above). This measures whether averaging N augmented views helps or hurts.

**Important:** these cells depend on the Phase 1 training cell (section 4) having run in this same session, because that's what created the checkpoint. If you restart the kernel, re-run sections 1–4 first.

## 9. Run standard TTA across N = 5, 10, 20, 50

In [ ]:
!python -m scripts.run_standard_tta \
    --dataset {DATASET} \
    --n-views 5 10 20 50 \
    --batch-size {BATCH_SIZE} \
    --seed {SEED} \
    --num-workers 2

## 10. Show the TTA results table

One row per N (plus the N=1 no-TTA baseline). Compare the `accuracy` column across rows — if it drops as N grows, TTA is hurting this dataset.

In [ ]:
import pandas as pd
tta_df = pd.read_csv(f'results/{DATASET}_standard_tta.csv')
print(tta_df.to_string(index=False))

## 11. Show the Accuracy-vs-N plot inline

Blue line = accuracy, red dashed = ECE, gray dashed = no-TTA baseline. If the blue line trends below the gray line as N increases, standard TTA is degrading this dataset — exactly the finding the paper is documenting.

In [ ]:
from IPython.display import Image, display
display(Image(filename=f'figures/tta/{DATASET}_accuracy_vs_n.png'))

## 12. Download artifacts

On **Kaggle**, the files appear under the right-side **Output** panel. On **Colab**, run the cell below.

Phase 1 produces: the checkpoint, baseline JSON, training log, reliability diagram, training curves.
Phase 2 adds: the standard-TTA CSV and the accuracy-vs-N plot.

In [ ]:
# COLAB ONLY — uncomment to download everything (Phase 1 + Phase 2)
# from google.colab import files
# files.download(f'checkpoints/{DATASET}_resnet18.pth')
# files.download(f'results/{DATASET}_baseline.json')
# files.download(f'results/{DATASET}_train_log.csv')
# files.download(f'results/{DATASET}_standard_tta.csv')
# files.download(f'figures/reliability/{DATASET}_baseline.png')
# files.download(f'figures/curves/{DATASET}_train_curves.png')
# files.download(f'figures/tta/{DATASET}_accuracy_vs_n.png')

---

## Next steps for the team

**Phase 1 deliverable:**
1. Paste the baseline tracker row into Sheet 1️⃣ Baselines.
2. Upload the `.pth` checkpoint to the shared Drive folder.

**Phase 2 deliverable:**
3. Paste the rows of `results/{DATASET}_standard_tta.csv` into Sheet 2️⃣ Standard TTA.
4. Note in the group chat whether TTA helped or hurt your dataset (the script prints a verdict).

**Then open a PR** with your results files. From your local clone:
```bash
git checkout -b s2-{dataset}-results   # adjust to your S-id and dataset
git add results/{dataset}_baseline.json results/{dataset}_train_log.csv results/{dataset}_standard_tta.csv
git commit -m "S2: {dataset} baseline + standard TTA results"
git push -u origin s2-{dataset}-results
```
Then open the PR on GitHub. @mustafaerensoyhan will review and merge.

**Don't commit:** the `.pth` checkpoint (gitignored, share via Drive), dataset `.npz` files (gitignored), or anything outside `results/`.